# Code Generation with Llama (Groq)

**Author:** Ibrahim  
**Environment:** Google Colab / Python 3

## Overview
This notebook uses Groq’s Llama 3.3 70B to generate code from natural language descriptions. It supports multiple programming languages (Python, JavaScript, SQL, Bash) and optionally validates syntax and executes Python code in a sandbox. This demonstrates an LLM as a programming assistant.

## What You Will Build
- A code generation function that takes a task description and language.
- Syntax validation for Python (using `ast`).
- Sandboxed execution of generated Python code.
- Interactive loop to test your own coding tasks.

## Why This Matters
LLMs that can write code are transforming software development. This project proves you can build a tool that automatically translates requirements into working code.

## Requirements
- **Groq API key** (free from [console.groq.com](https://console.groq.com))

---

**© 2026 Ibrahim – Code generation with Llama.**

### Install & Imports

In [1]:
!pip install -q groq

import re
import ast
import sys
from io import StringIO
from getpass import getpass
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.0 MB/s eta 0:00:00


### API Key & Groq Client

In [2]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"
print("Groq client ready.")

Enter your Groq API key: ··········
Groq client ready.


### Code Generation Function

In [3]:
def generate_code(description, language="python", max_tokens=1024):
    prompt = f"""You are an expert programmer. Generate only the code, no explanations.

Language: {language}
Task: {description}
Code:
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=max_tokens
    )
    code = response.choices[0].message.content.strip()
    # Remove markdown code blocks if present
    code = re.sub(r"```\w*\n?|```", "", code)
    return code

### yntax Validation (Python)

In [4]:
def validate_python_syntax(code):
    try:
        ast.parse(code)
        return True, "Valid syntax"
    except SyntaxError as e:
        return False, str(e)

### Sandboxed Execution (Python only)

In [5]:
def execute_python_sandbox(code, timeout=5):
    """Execute Python code in a restricted environment."""
    # Redirect stdout
    old_stdout = sys.stdout
    sys.stdout = captured = StringIO()
    try:
        # Limited globals for safety
        safe_globals = {"__builtins__": {"print": print, "range": range, "len": len, "int": int, "str": str, "float": float, "list": list, "dict": dict}}
        exec(code, safe_globals, {})
        output = captured.getvalue()
        return True, output if output else "Code executed successfully (no output)."
    except Exception as e:
        return False, str(e)
    finally:
        sys.stdout = old_stdout

### Full Pipeline

In [6]:
def generate_and_test(description, language="python", execute=False):
    print(f" Task: {description}")
    print(f" Language: {language}")
    code = generate_code(description, language)
    print("\n Generated Code:\n")
    print(code)

    if language.lower() == "python":
        valid, msg = validate_python_syntax(code)
        print(f" Syntax check: {msg}")
        if execute and valid:
            success, output = execute_python_sandbox(code)
            print(f"▶ Execution: {' Success' if success else ' Failed'}")
            if output:
                print(f" Output:\n{output}")
    return code

### Test Examples

In [7]:
examples = [
    ("Write a Python function to compute fibonacci numbers recursively", "python"),
    ("Write a SQL query to select all customers who bought more than 5 items", "sql"),
    ("Write a JavaScript function that returns the factorial of a number", "javascript"),
]

for desc, lang in examples:
    generate_and_test(desc, lang, execute=False)

 Task: Write a Python function to compute fibonacci numbers recursively
 Language: python

 Generated Code:

def fibonacci(n):
    if n <= 0:
        return "Input should be a positive integer"
    elif n == 1:
        return 0
    elif n == 2:
        return 1
    else:
        return fibonacci(n-1) + fibonacci(n-2)

def main():
    n = 10  # example input
    print(f"The {n}th Fibonacci number is: {fibonacci(n)}")

if __name__ == "__main__":
    main()

 Syntax check: Valid syntax
 Task: Write a SQL query to select all customers who bought more than 5 items
 Language: sql

 Generated Code:

SELECT c.customer_id, c.customer_name, SUM(oi.quantity) AS total_items
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.customer_name
HAVING SUM(oi.quantity) > 5;

 Task: Write a JavaScript function that returns the factorial of a number
 Language: javascript

 Generated Code:

function factorial(n) {
    if (

### Interactive Loop

In [9]:
print("\nCode Generator Ready. Type 'exit' to quit.")
while True:
    desc = input("\n Describe the code you need: ").strip()
    if desc.lower() == "exit":
        break
    if not desc:
        continue
    lang = input("🗣️ Language (python/sql/javascript/bash) [python]: ").strip() or "python"
    exec_choice = input(" Execute Python code? (y/n) [n]: ").strip().lower()
    generate_and_test(desc, lang, execute=(exec_choice == 'y'))


Code Generator Ready. Type 'exit' to quit.

 Describe the code you need: write a python code which would generate the table for the number user gonna input 
🗣️ Language (python/sql/javascript/bash) [python]: python 
 Execute Python code? (y/n) [n]: n
 Task: write a python code which would generate the table for the number user gonna input
 Language: python

 Generated Code:

def generate_table(n):
    for i in range(1, 11):
        print(f"{n} x {i} = {n * i}")

def main():
    num = int(input("Enter a number: "))
    print(f"Table of {num}:")
    generate_table(num)

if __name__ == "__main__":
    main()

 Syntax check: Valid syntax

 Describe the code you need: exit


### Final Summary

In [10]:
print("Code Generation with Llama - COMPLETED")
print("Author: Ibrahim")
print(" Generate code in multiple languages.")
print(" Syntax validation for Python.")
print(" Sandboxed execution (optional).")
print(" Ready to integrate into developer tools.")

Code Generation with Llama - COMPLETED
Author: Ibrahim
 Generate code in multiple languages.
 Syntax validation for Python.
 Sandboxed execution (optional).
 Ready to integrate into developer tools.
